<a href="https://colab.research.google.com/github/hmnaz213/native/blob/main/alibaba_shipping_forwarder_order_management.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Load Google Sheet Data

In [11]:
import pandas as pd
import gspread

In [12]:
import pandas as pd
import gspread
from google.colab import auth
from google.auth import default
from datetime import datetime

try:
    # Authenticate user with Google Colab
    auth.authenticate_user()

    # Get default credentials
    creds, _ = default()

    # Authorize gspread with the obtained credentials
    gc = gspread.authorize(creds)
    print("Authenticated successfully with Google Colab.")

    spreadsheet_url = 'https://docs.google.com/spreadsheets/d/13rC6ZVcKKeHAeYoJPi86s-6VO6g0k4cRkGWzBeGwvlo/edit?gid=463685144#gid=463685144'

    # The GID is provided in the URL and is 463685144
    worksheet_gid = 463685144

    # Open the spreadsheet by URL
    sh = gc.open_by_url(spreadsheet_url)

    # Select the worksheet by its gid
    worksheet = sh.get_worksheet_by_id(worksheet_gid)

    # Get all values from the worksheet as a list of lists
    data = worksheet.get_all_values()

    # Convert to a pandas DataFrame. The first row is assumed to be the header.
    if data:
        # Clean header: remove any empty strings that might be trailing
        header = [h for h in data[0] if h]
        num_columns = len(header)

        # Helper function to check if a string can be parsed as a date
        def is_date(s, format_str='%d/%m/%Y'):
            if not isinstance(s, str):
                return False
            try:
                datetime.strptime(s, format_str)
                return True
            except (ValueError, TypeError):
                return False

        processed_data = []
        for row_idx, row in enumerate(data[1:]):
            current_row = row[:] # Make a copy to modify

            # Heuristic 1: Check for leading shift (e.g., 'ssot/alibaba_order_creation' in email_date)
            # This applies if the first element is NOT a date but the second is.
            if len(current_row) > 1 and not is_date(current_row[0]) and is_date(current_row[1]):
                current_row = current_row[1:]

            # Heuristic 2: Check for '0' in the 'ingested_at' position, shifting valid data right
            # 'ingested_at' is typically the 9th column (index 8) after a clean header of 10.
            # If current_row[8] is '0' and current_row[9] is a valid date (expected for ingested_at)
            # then remove the '0' at index 8.
            if (len(current_row) > 8 and current_row[8] == '0' and
                len(current_row) > 9 and is_date(current_row[9])):
                current_row.pop(8) # Remove the '0'

            # Final step: Normalize row length to match the clean header, padding or truncating
            if len(current_row) < num_columns:
                current_row = current_row + [None] * (num_columns - len(current_row))
            elif len(current_row) > num_columns:
                current_row = current_row[:num_columns]

            processed_data.append(current_row)

        df = pd.DataFrame(processed_data, columns=header)
        print("Google Sheet loaded successfully into DataFrame 'df'.")
        print("First 5 rows of the DataFrame (after improved alignment):")
        display(df.head())
    else:
        print("The specified worksheet is empty. No data to load.")

except Exception as e:
    print(f"An error occurred during Google Sheet loading: {e}")
    print("Please ensure you have authenticated successfully, the spreadsheet URL and worksheet GID are correct, and the service account has appropriate permissions.")

Authenticated successfully with Google Colab.
Google Sheet loaded successfully into DataFrame 'df'.
First 5 rows of the DataFrame (after improved alignment):


,email_date,sender,subject,gmail_message_id,thread_id,raw_body,raw_snippet,has_attachments,ingested_at,parse_status
0,13/05/2026,Alibaba <credit@notice.alibaba.com>,【Action Required】Your Trade Assurance Order No...,19e1ef8bab06e47f,19e1ef8bab06e47f,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,Delivered-To: hmnaz213@gmail.com\r\nReceived: ...,FALSE,16/05/2026,raw_ingested
1,16/04/2026,Alibaba <credit@notice.alibaba.com>,【Action Required】Your Trade Assurance Order No...,19d9432ae9a644eb,19d9432ae9a644eb,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,FALSE,16/05/2026,raw_ingested
2,14/04/2026,"""Alibaba.com"" <credit@notice.alibaba.com>",【Action Required】Your Trade Assurance order 29...,19d8ba45cbe3e605,19d8ba45cbe3e605,\nTrade Assurance\n【Action Required】Your Trade...,\nTrade Assurance\n【Action Required】Your Trade...,FALSE,16/05/2026,raw_ingested
3,14/04/2026,Alibaba <credit@notice.alibaba.com>,【Action Required】Your Trade Assurance Order No...,19d8ba41ee2d475d,19d8ba41ee2d475d,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,FALSE,16/05/2026,raw_ingested
4,08/04/2026,Alibaba <credit@notice.alibaba.com>,【Action Required】Your Trade Assurance Order No...,19d6d2cf58c0e570,19d6d2cf58c0e570,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,FALSE,16/05/2026,raw_ingested


In [13]:
import re

# Define the regex pattern to find the order number
# It looks for the specific phrase and then captures digits that follow it.
order_pattern = r'【Action Required】Your Trade Assurance Order No\. (\d+)'

# Apply the regex to the 'subject' column to extract the order numbers
# .str.extract() returns a DataFrame with one column for each captured group
order_numbers_df = df['subject'].str.extract(order_pattern)

# Rename the column for clarity
order_numbers_df.columns = ['order_number']

print("Extracted Order Numbers DataFrame:")
display(order_numbers_df.head())

Extracted Order Numbers DataFrame:


,order_number
0,300968051001028908
1,298896450001028908
2,NaN
3,297668291501028908
4,297008607501028908


In [14]:
# Display information about the DataFrame to confirm the parsing on the entire dataset
# print("Information about the extracted order numbers DataFrame:")
# order_numbers_df.info()

# Combine the original DataFrame with the extracted order numbers
df_with_order_numbers = pd.concat([df, order_numbers_df], axis=1)

print("Original DataFrame with extracted order numbers added:")
display(df_with_order_numbers.head())

Original DataFrame with extracted order numbers added:


,email_date,sender,subject,gmail_message_id,thread_id,raw_body,raw_snippet,has_attachments,ingested_at,parse_status,order_number
0,13/05/2026,Alibaba <credit@notice.alibaba.com>,【Action Required】Your Trade Assurance Order No...,19e1ef8bab06e47f,19e1ef8bab06e47f,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,Delivered-To: hmnaz213@gmail.com\r\nReceived: ...,FALSE,16/05/2026,raw_ingested,300968051001028908
1,16/04/2026,Alibaba <credit@notice.alibaba.com>,【Action Required】Your Trade Assurance Order No...,19d9432ae9a644eb,19d9432ae9a644eb,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,FALSE,16/05/2026,raw_ingested,298896450001028908
2,14/04/2026,"""Alibaba.com"" <credit@notice.alibaba.com>",【Action Required】Your Trade Assurance order 29...,19d8ba45cbe3e605,19d8ba45cbe3e605,\nTrade Assurance\n【Action Required】Your Trade...,\nTrade Assurance\n【Action Required】Your Trade...,FALSE,16/05/2026,raw_ingested,NaN
3,14/04/2026,Alibaba <credit@notice.alibaba.com>,【Action Required】Your Trade Assurance Order No...,19d8ba41ee2d475d,19d8ba41ee2d475d,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,FALSE,16/05/2026,raw_ingested,297668291501028908
4,08/04/2026,Alibaba <credit@notice.alibaba.com>,【Action Required】Your Trade Assurance Order No...,19d6d2cf58c0e570,19d6d2cf58c0e570,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,FALSE,16/05/2026,raw_ingested,297008607501028908
